# 01 - Ingesta de documentos

En este cuaderno leemos los documentos de data/raw/, uno por formato, y los convertimos en texto plano. Después partimos ese texto en trozos pequeños (chunks) y los guardamos en data/processed/chunks.csv. Esos trozos son los que el asistente buscará más adelante.

Formatos: PDF, HTML, XML, CSV y Markdown.

# Importaciones

- **pypdf**: lee el texto de los PDF.
- **BeautifulSoup**: lee HTML y XML y permite quedarse solo con el texto, sin etiquetas.
- **pandas**: para el CSV y para guardar los trozos al final.

In [1]:
import pandas as pd
from pypdf import PdfReader
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

Guardaremos cada documento leído en una lista llamada documentos. Cada elemento será un diccionario con dos datos: el texto y la fuente (el nombre del archivo), para saber después de dónde sale cada respuesta.

In [2]:
documentos = []

# PDF

PdfReader abre el PDF y recorremos sus páginas una a una, extrayendo el texto de cada una con extract_text() y juntándolo todo.

In [3]:
archivos_pdf = ['rd_1457_1986_talleres_reparacion.pdf', 'dudas_anexo_ix_reglamento_vehiculos.pdf']

for archivo in archivos_pdf:
    reader = PdfReader('../data/raw/' + archivo)
    texto = ''
    for pagina in reader.pages:
        texto = texto + pagina.extract_text() + '\n'
    documentos.append({'fuente': archivo, 'texto': texto})
    print(archivo, len(reader.pages), 'páginas,', len(texto), 'caracteres')

rd_1457_1986_talleres_reparacion.pdf 16 páginas, 53184 caracteres
dudas_anexo_ix_reglamento_vehiculos.pdf 31 páginas, 35120 caracteres


# HTML

BeautifulSoup interpreta la página web. Quitamos las partes que no son contenido (scripts, estilos, menús) y con get_text() nos quedamos solo con el texto.

In [4]:
archivos_html = ['rd_920_2017_inspeccion_tecnica_vehiculos.html', 'dgt_llamadas_a_revision.html']

for archivo in archivos_html:
    with open('../data/raw/' + archivo, encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    for etiqueta in soup(['script', 'style', 'nav', 'header', 'footer']):
        etiqueta.decompose()
    texto = soup.get_text(separator='\n', strip=True)
    documentos.append({'fuente': archivo, 'texto': texto})
    print(archivo, len(texto), 'caracteres')

rd_920_2017_inspeccion_tecnica_vehiculos.html 200735 caracteres
dgt_llamadas_a_revision.html 6350 caracteres


# XML

El XML del BOE tiene metadatos, un análisis con referencias a otras normas y, al final, el texto de la norma en la etiqueta texto que cuelga directamente de documento. Con recursive=False le decimos a BeautifulSoup que busque solo en ese primer nivel, para no coger los textos de las referencias.

In [5]:
archivo = 'rd_920_2017_inspeccion_tecnica_vehiculos.xml'
with open('../data/raw/' + archivo, encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'xml')

titulo = soup.find('titulo').get_text()
texto = titulo + '\n' + soup.find('documento').find('texto', recursive=False).get_text(separator='\n', strip=True)
documentos.append({'fuente': archivo, 'texto': texto})
print(archivo, len(texto), 'caracteres')

rd_920_2017_inspeccion_tecnica_vehiculos.xml 192092 caracteres


# CSV

El CSV es una tabla de estaciones ITV. Un buscador de texto entiende mejor frases que columnas sueltas, así que convertimos cada fila en una frase del tipo: La estación ITV 411 de VEIASA está en HUERCAL DE ALMERIA (Almería, Andalucía)...

In [6]:
df = pd.read_csv('../data/raw/registro_estaciones_itv_2026.csv')
df.head()

,RAZÓN SOCIAL,CÓDIGO ESTACIÓN,RÉGIMEN,COM AUTON,PROVINCIA,MUNICIPIO,DIRECCIÓN POSTAL,C. P.,L.T.,LIGEROS,PESADOS,UNIVERSALES,ESPECIALES,SUSPENSIÓN INICIO,EXENTA ACREDITACIÓN,TIPO DE INSPECCIONES
0,VEIASA,411,Empresa Pública,Andalucía,Almería,HUERCAL DE ALMERIA,"Paraje de la Cepa, s/n",4230,6.0,4.0,0.0,0.0,2.0,NaN,NaN,NaN
1,VEIASA,412,Empresa Pública,Andalucía,Almería,HUERCAL DE ALMERIA,"Ctra N 340 KM 121 Paraje de Zaramula, s/n.",4230,3.0,1.0,0.0,2.0,0.0,NaN,NaN,NaN
2,VEIASA,413,Empresa Pública,Andalucía,Almería,VICAR,Ctra. Nacional N-340 (Pol. Ind. Venta El Cosar...,4738,4.0,2.0,0.0,2.0,0.0,NaN,NaN,NaN
3,VEIASA,414,Empresa Pública,Andalucía,Almería,NIJAR,"Pg. La Granatilla, Campohermoso. C/ Tornero, 42",4230,2.0,1.0,0.0,1.0,0.0,NaN,NaN,NaN
4,VEIASA,421,Empresa Pública,Andalucía,Almería,VERA,Autovía del Mediterráneo - Acceso Sur a Vera -...,4620,2.0,1.0,0.0,1.0,0.0,NaN,NaN,NaN


In [7]:
df.shape

(730, 16)

In [8]:
frases = []
for i, fila in df.iterrows():
    frase = ('La estación ITV ' + str(fila['CÓDIGO ESTACIÓN']) + ' de ' + str(fila['RAZÓN SOCIAL'])
             + ' (' + str(fila['RÉGIMEN']) + ') está en ' + str(fila['DIRECCIÓN POSTAL']) + ', '
             + str(fila['MUNICIPIO']) + ' (' + str(fila['PROVINCIA']) + ', ' + str(fila['COM AUTON']) + ').')
    frases.append(frase)

texto = '\n'.join(frases)
documentos.append({'fuente': 'registro_estaciones_itv_2026.csv', 'texto': texto})
print(frases[0])

La estación ITV 411 de VEIASA (Empresa Pública) está en Paraje de la Cepa, s/n, HUERCAL DE ALMERIA (Almería, Andalucía).


# Markdown

El Markdown ya es texto plano, así que basta con leerlo tal cual.

In [9]:
archivo = 'faq_taller.md'
with open('../data/raw/' + archivo, encoding='utf-8') as f:
    texto = f.read()
documentos.append({'fuente': archivo, 'texto': texto})
print(archivo, len(texto), 'caracteres')

faq_taller.md 4982 caracteres


# Chunking (partir en trozos)

Los documentos son demasiado largos para dárselos enteros al modelo, así que los partimos en trozos de 800 caracteres. Cada trozo empieza 100 caracteres antes de donde acabó el anterior (solape), para no cortar una idea justo por la mitad y perderla.

Es el método más sencillo: por número de caracteres.

In [10]:
tamano = 800
solape = 100

trozos = []
for doc in documentos:
    texto = doc['texto']
    inicio = 0
    while inicio < len(texto):
        trozo = texto[inicio:inicio + tamano]
        trozos.append({'fuente': doc['fuente'], 'texto': trozo})
        inicio = inicio + tamano - solape

len(trozos)

850

# Resumen de trozos por documento

In [11]:
df_trozos = pd.DataFrame(trozos)
df_trozos['fuente'].value_counts()

fuente
rd_920_2017_inspeccion_tecnica_vehiculos.html    287
rd_920_2017_inspeccion_tecnica_vehiculos.xml     275
registro_estaciones_itv_2026.csv                 143
rd_1457_1986_talleres_reparacion.pdf              76
dudas_anexo_ix_reglamento_vehiculos.pdf           51
dgt_llamadas_a_revision.html                      10
faq_taller.md                                      8
Name: count, dtype: int64

In [12]:
df_trozos.head()

,fuente,texto
0,rd_1457_1986_talleres_reparacion.pdf,"Real Decreto 1457/1986, de 10 de enero, por el..."
1,rd_1457_1986_talleres_reparacion.pdf,................ 4\nArtículo 3. Clasificación ...
2,rd_1457_1986_talleres_reparacion.pdf,..................... 7\nTÍTULO III. Centros d...
3,rd_1457_1986_talleres_reparacion.pdf,a. ........................................ 10...
4,rd_1457_1986_talleres_reparacion.pdf,............... 13\nDisposición transitoria se...


# Guardar

Guardamos los trozos en data/processed/chunks.csv para usarlos en el siguiente cuaderno.

In [13]:
df_trozos.to_csv('../data/processed/chunks.csv', index=False)
print('Guardados', len(df_trozos), 'trozos')

Guardados 850 trozos


# Observaciones

- Hemos leído los 5 formatos y los hemos convertido en texto con su fuente.
- Cada fila del CSV se ha convertido en una frase para que el buscador la entienda.
- Los trozos son de 800 caracteres con 100 de solape.